# Exercise 05 — RESTful Web Services: the Client
### 🌐 Talking to the SmartCity API

In the previous exercises you built sensor data in memory.
In the real world, that data lives on a **server** and you fetch it over HTTP.

**REST** (Representational State Transfer) maps the four basic data operations
onto standard HTTP methods:

| HTTP method | CRUD action | Example |
|---|---|---|
| `GET` | Read | Fetch all sensors |
| `POST` | Create | Register a new sensor |
| `PUT` | Update/Replace | Update a sensor reading |
| `DELETE` | Delete | Remove a sensor |

In this exercise you play the **client** role using the `requests` library.
We will talk to a real public REST API first, then to our own local server.

---

## 1️⃣  The `requests` library — your HTTP Swiss Army knife

In [ ]:
import requests
import json

# GET — retrieve a resource
# httpbin.org is a free public service for testing HTTP requests
response = requests.get("https://httpbin.org/get")

print("Status code :", response.status_code)   # 200 = OK
print("Content-Type:", response.headers["Content-Type"])
print()
data = response.json()   # parse response body as JSON
print("My IP address:", data["origin"])

## 2️⃣  HTTP status codes — always check them!

The server communicates success or failure through status codes.
As your lecture notes explain: `2xx` = success, `4xx` = client error, `5xx` = server error.

In [ ]:
def check_response(r):
    """Print a friendly summary of an HTTP response."""
    if r.status_code == 200:
        print(f"✅ 200 OK")
    elif r.status_code == 201:
        print(f"✅ 201 Created — resource was added")
    elif r.status_code == 404:
        print(f"❌ 404 Not Found — resource does not exist")
    elif r.status_code == 400:
        print(f"❌ 400 Bad Request — your request had an error")
    else:
        print(f"ℹ️  {r.status_code}")

# Test a page that exists
check_response(requests.get("https://httpbin.org/status/200"))
# Test a page that doesn't exist
check_response(requests.get("https://httpbin.org/status/404"))
# Test a bad request
check_response(requests.get("https://httpbin.org/status/400"))

## 3️⃣  Start a local SmartCity API server

We use Flask to start a tiny REST server **in a background thread**.
This means both server and client live in the same notebook — no terminal needed!

> 💡 This is exactly what your lecture describes: a device running a lightweight webserver
> that clients (browsers, other devices) can talk to over HTTP.

In [ ]:
import threading
from flask import Flask, jsonify, request

# ── In-memory data store (our "database") ──────────────────────────────────
SENSORS = {
    "TRN-001": {"sensor_id": "TRN-001", "location": "Piazza Castello",  "temperature_c": 22.4, "pm25_ugm3": 12.1},
    "TRN-002": {"sensor_id": "TRN-002", "location": "Piazza Vittorio",  "temperature_c": 23.1, "pm25_ugm3": 18.4},
    "TRN-003": {"sensor_id": "TRN-003", "location": "Mirafiori",        "temperature_c": 21.0, "pm25_ugm3": 35.2},
    "TRN-004": {"sensor_id": "TRN-004", "location": "Lingotto",         "temperature_c": 24.5, "pm25_ugm3": 22.7},
    "TRN-005": {"sensor_id": "TRN-005", "location": "Parco del Po",     "temperature_c": 20.8, "pm25_ugm3":  8.3},
}

app = Flask(__name__)

@app.route("/sensors", methods=["GET"])
def get_all_sensors():
    return jsonify(list(SENSORS.values()))

@app.route("/sensors/<sensor_id>", methods=["GET"])
def get_sensor(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": "sensor not found"}), 404
    return jsonify(SENSORS[sensor_id])

@app.route("/sensors", methods=["POST"])
def add_sensor():
    body = request.get_json()
    if not body or "sensor_id" not in body:
        return jsonify({"error": "sensor_id is required"}), 400
    sid = body["sensor_id"]
    SENSORS[sid] = body
    return jsonify(body), 201

@app.route("/sensors/<sensor_id>", methods=["PUT"])
def update_sensor(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": "sensor not found"}), 404
    SENSORS[sensor_id].update(request.get_json())
    return jsonify(SENSORS[sensor_id])

@app.route("/sensors/<sensor_id>", methods=["DELETE"])
def delete_sensor(sensor_id):
    if sensor_id not in SENSORS:
        return jsonify({"error": "sensor not found"}), 404
    deleted = SENSORS.pop(sensor_id)
    return jsonify({"deleted": deleted["sensor_id"]})

# Start server in background
import logging
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)   # silence Flask's request logs

t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
t.daemon = True
t.start()

import time; time.sleep(1)   # give the server a moment to start
print("🚀 SmartCity API running at http://localhost:5000")
print("   GET  /sensors           → all sensors")
print("   GET  /sensors/<id>      → one sensor")
print("   POST /sensors           → add a sensor")
print("   PUT  /sensors/<id>      → update a sensor")
print("   DELETE /sensors/<id>    → remove a sensor")

## 4️⃣  GET — read resources

In [ ]:
BASE = "http://localhost:5000"

# Get all sensors
r = requests.get(f"{BASE}/sensors")
sensors = r.json()
print(f"GET /sensors → {r.status_code}")
for s in sensors:
    print(f"  {s['sensor_id']}  {s['location']:20s}  {s['temperature_c']}°C")

In [ ]:
# Get one sensor by ID
r = requests.get(f"{BASE}/sensors/TRN-003")
print(f"GET /sensors/TRN-003 → {r.status_code}")
print(json.dumps(r.json(), indent=2))

In [ ]:
# Ask for a sensor that doesn't exist
r = requests.get(f"{BASE}/sensors/TRN-999")
print(f"GET /sensors/TRN-999 → {r.status_code}")   # expect 404
print(r.json())

## 5️⃣  POST — create a new resource

In [ ]:
new_sensor = {
    "sensor_id": "TRN-006",
    "location": "Porta Nuova",
    "temperature_c": 21.5,
    "pm25_ugm3": 14.0
}

r = requests.post(f"{BASE}/sensors", json=new_sensor)   # json= sets Content-Type automatically
print(f"POST /sensors → {r.status_code}")   # expect 201 Created
print(json.dumps(r.json(), indent=2))

## 6️⃣  PUT — update an existing resource

In [ ]:
# The Lingotto sensor just sent a new reading
r = requests.put(f"{BASE}/sensors/TRN-004", json={"temperature_c": 26.1, "pm25_ugm3": 19.0})
print(f"PUT /sensors/TRN-004 → {r.status_code}")
print(json.dumps(r.json(), indent=2))

## 7️⃣  DELETE — remove a resource

In [ ]:
r = requests.delete(f"{BASE}/sensors/TRN-005")
print(f"DELETE /sensors/TRN-005 → {r.status_code}")
print(r.json())

# Verify it's gone
remaining = requests.get(f"{BASE}/sensors").json()
print(f"\nSensors remaining: {len(remaining)}")

---
## 🏆 Challenges

1. Write a function `get_sensor_safe(sensor_id)` that returns the sensor dict or `None` if 404, without raising an exception.
2. Register all 5 sensors from Exercise 01 into the running server using a loop of POST requests.
3. Fetch all sensors, filter those with `pm25_ugm3 > 20`, and print a pollution alert for each.
4. Write a function `update_all_temperatures(delta)` that fetches all sensors and PUTs each one back with temperature increased by `delta`.

In [ ]:
# Your code here
